# PhysIR vs DarkIR-m (released weights) — direct comparison · Kaggle **CPU**

Evaluates the authors' released **DarkIR-m** checkpoint and our checkpoints (**PhysIR full**,
**abl_percep@384**) under ONE pipeline: same test loader, same metrics, **both LPIPS
conventions** per image (`lpips` = correct [−1,1]; `lpips_darkirconv` = DarkIR's unscaled
[0,1] eval, their `test_utils.py` L57). Ends with the comparison table + paired Wilcoxon
(PhysIR vs actual DarkIR-m — guide G6).

**Inputs to attach:** `physir-code`, `lolblur-dataset`, your dataset(s)/model(s) containing
`DarkIR_384.pt` + your run checkpoints (`stage4_physir_final_latest.pt`, `stage5_percep_latest.pt`
or similar), and (optionally) your existing `per_image_*.csv` outputs — any layout works,
discovery is by filename.

**CPU + resume:** ~8–15 s/img → ~5–7 h per model. CSVs are written **incrementally** (flush
per image) and every re-run resumes from the last written row — so just re-run across sessions
(CPU burns no GPU quota; save a version before the 12 h cap, attach it next session).

**Validation gate:** DarkIR-m released must reproduce **≈27.00 PSNR / 0.883 SSIM** under our
loader — that simultaneously authenticates the checkpoint and validates the pipeline. If it
doesn't match, STOP and investigate before trusting any comparison.

**Fairness note for writing:** their model trained on all 10,200 train pairs; ours on 90%.


## 1. Install

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops

## 2. Discover code, data, checkpoints, and existing per-image CSVs

In [ ]:
import os, sys, glob
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'
WORKING_DIR  = '/kaggle/working'

def find_root(base, marker, depth=3):
    if os.path.exists(os.path.join(base, marker)): return base
    for d in range(1, depth+1):
        h = sorted(glob.glob(os.path.join(base, *(['*']*d), marker)))
        if h: return h[0][:-len(marker)].rstrip(os.sep)
    return None

CODE_ROOT = find_root(CODE_DIR, 'train.py') or find_root('/kaggle/input', 'train.py', 4)
assert CODE_ROOT, 'physir-code not found'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)
HAVE_WILCOXON = os.path.isfile(os.path.join(CODE_ROOT, 'paired_wilcoxon.py'))
if not HAVE_WILCOXON:
    print('NOTE: paired_wilcoxon.py missing from physir-code — upload it as a single file '
          'to the dataset (repo root) to enable the significance cell.')

cand = find_root(LOLBLUR_ROOT, os.path.join('test', 'low_blur_noise')) or \
       find_root('/kaggle/input', os.path.join('test', 'low_blur_noise'), 4)
assert cand, 'LOLBlur test not found'
LOLBLUR = cand
print('CODE_ROOT:', CODE_ROOT, '| wilcoxon:', HAVE_WILCOXON)
print('LOLBLUR  :', LOLBLUR)

# ── discover checkpoints anywhere under /kaggle/input (skip macOS ._ junk) ─────
pts = []
for d in range(1, 5):
    pts += glob.glob(os.path.join('/kaggle/input', *(['*']*d), '*.pt'))
pts = [p for p in pts if not os.path.basename(p).startswith('._')]

def classify(path):
    b = os.path.basename(path).lower()
    if 'physir' in b or 'stage4' in b:                  return 'physir_full'
    if 'percep' in b:                                   return 'abl_percep384'
    if 'darkir' in b:                                   return 'darkir_m_released'
    return None

MODELS = {}
for p in sorted(pts):
    lab = classify(p)
    if lab is None: continue
    # prefer *_latest.pt when several files map to one label (ep300 EMA lives there)
    if lab not in MODELS or ('latest' in os.path.basename(p).lower()
                             and 'latest' not in os.path.basename(MODELS[lab]).lower()):
        MODELS[lab] = p

print('\nMODEL MAP (label -> checkpoint):')
for k, v in MODELS.items(): print(f'  {k:18s} {v}')
unmatched = [p for p in pts if classify(p) is None]
if unmatched: print('\nunmatched .pt files (rename to include darkir/physir/percep):', unmatched)
assert 'darkir_m_released' in MODELS, 'DarkIR_384.pt not found among inputs'

# ── existing per-image CSVs (your uploaded outputs) → working copy ─────────────
PERIMG = os.path.join(WORKING_DIR, 'per_image'); os.makedirs(PERIMG, exist_ok=True)
import shutil
found = []
for d in range(1, 5):
    found += glob.glob(os.path.join('/kaggle/input', *(['*']*d), 'per_image_*.csv'))
for src in found:
    b = os.path.basename(src)
    if b.startswith('._'): continue
    dst = os.path.join(PERIMG, b)
    if not os.path.exists(dst): shutil.copy2(src, dst)
print('\nper-image CSVs available:', sorted(os.listdir(PERIMG)))

## 3. EVAL — one pipeline, both LPIPS conventions, incremental + resumable

In [ ]:
import torch, csv, time
import numpy as np
torch.set_num_threads(os.cpu_count() or 4)
import train as T
from data import create_test_data
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'running on: {device}' + ('' if device=='cuda' else '  (CPU session — ~6 h/model; GPU cuts this to ~25-35 min/model)'))

opt_test = {'name':'LOLBlur','val':{'test_path': os.path.join(LOLBLUR,'test'), 'batch_size_test':1},
            'train':{'n_workers':2,'verbose':True}}
tl_raw, _ = create_test_data(rank=0, world_size=1, opt=opt_test)
tl = tl_raw if not isinstance(tl_raw, dict) else list(tl_raw.values())[0]['loader']
N_TEST = len(tl)

assert T._LPIPS_AVAILABLE, 'lpips missing'
lpips_fn = T._LPIPS(net='vgg', verbose=False).to(device)
T._ssim_metric.to(device)

NET_CFG = {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
           'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

def load_model(ckpt_path):
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(ck, dict) and not all(hasattr(v, 'shape') for v in ck.values()):
        state = ck.get('ema') or ck.get('model') or ck.get('params')
        which = 'ema' if ck.get('ema') else ('model' if ck.get('model') else 'params')
    else:
        state, which = ck, 'raw'
    m = T.build_model(NET_CFG).to(device)
    missing, unexpected = m.load_state_dict(
        {k.replace('module.',''): v for k, v in state.items()}, strict=False)
    print(f'  weights: {which} | missing {len(missing)} unexpected {len(unexpected)}')
    assert len(missing) < 5, f'too many missing keys — wrong architecture? {missing[:5]}'
    m.eval(); return m

REFS = {'darkir_m_released': (27.00, 0.883, 'published DarkIR-m (Table 1)'),
        'physir_full':       (27.286, 0.8838, 'our Stage 4 GPU eval'),
        'abl_percep384':     (26.927, 0.8713, 'our stage5_percep GPU eval')}

for label, ckpt in MODELS.items():
    out_csv = os.path.join(PERIMG, f'per_image_{label}.csv')
    n_done = 0
    if os.path.exists(out_csv):
        n_done = max(0, sum(1 for _ in open(out_csv)) - 1)
    if n_done >= N_TEST:
        print(f'✅ {label}: {n_done}/{N_TEST} — done'); continue
    print(f'\n===== {label}  (resume at {n_done}/{N_TEST}) =====')
    model = load_model(ckpt)
    mode = 'a' if n_done else 'w'
    f = open(out_csv, mode, newline='')
    w = csv.writer(f)
    if mode == 'w': w.writerow(['idx','psnr','ssim','lpips','lpips_darkirconv']); f.flush()
    t0, n_new = time.time(), 0
    with torch.no_grad():
        for i, (high, low) in enumerate(tl):
            if i < n_done: continue
            pred = model(low, side_loss=False).clamp(0, 1)
            _, _, H, W = high.shape; pred = pred[:, :, :H, :W]
            mse  = torch.mean((pred - high) ** 2).item()
            psnr = 20 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))
            ssim = T._ssim_metric(pred, high).item()
            lp   = lpips_fn(pred*2-1, high*2-1).mean().item()   # correct [-1,1]
            lpdk = lpips_fn(pred, high).mean().item()           # DarkIR's unscaled convention
            w.writerow([i, psnr, ssim, lp, lpdk]); f.flush()
            n_new += 1
            if n_new % 50 == 0:
                r = (time.time()-t0)/n_new
                print(f'  {i+1}/{N_TEST} · {r:.1f}s/img · ~{r*(N_TEST-i-1)/3600:.1f}h left', flush=True)
    f.close()
    a = np.loadtxt(out_csv, delimiter=',', skiprows=1)
    ps, ss = a[:,1].mean(), a[:,2].mean()
    print(f'{label}: PSNR {ps:.4f} · SSIM {ss:.4f} · LPIPS {a[:,3].mean():.4f} · '
          f'LPIPS(DarkIR-conv) {a[:,4].mean():.4f}')
    if label in REFS:
        rp, rs, srcname = REFS[label]
        ok = abs(ps-rp) < 0.15 and abs(ss-rs) < 0.01
        print(f'  vs {srcname} ({rp}/{rs}): {"✅ MATCH" if ok else "⚠️ MISMATCH — investigate before comparing!"}')
    del model
print('\nEVAL complete (or resume next session).')

## 4. Comparison table

In [ ]:
import numpy as np, csv as _csv
def means(path):
    rows = list(_csv.DictReader(open(path)))
    g = lambda c: (np.mean([float(r[c]) for r in rows]) if c in rows[0] else None)
    return {'n':len(rows),'psnr':g('psnr'),'ssim':g('ssim'),'lpips':g('lpips'),'dk':g('lpips_darkirconv')}

print(f'{"model":24s}{"n":>6s}{"PSNR":>9s}{"SSIM":>9s}{"LPIPS":>9s}{"LPIPS(dk-conv)":>15s}')
print('-'*72)
for fn in sorted(os.listdir(PERIMG)):
    if not fn.startswith('per_image_'): continue
    m = means(os.path.join(PERIMG, fn))
    lab = fn[len('per_image_'):-4]
    dk = f"{m['dk']:.4f}" if m['dk'] is not None else '   —'
    print(f"{lab:24s}{m['n']:>6d}{m['psnr']:>9.4f}{m['ssim']:>9.4f}{m['lpips']:>9.4f}{dk:>15s}")
print('-'*72)
print('published DarkIR-m           27.0000   0.8830        —         0.1620  (their conv.)')
print('published DarkIR-l           27.3000   0.8980        —         0.1370  (their conv.)')
print("\nNote: compare 'LPIPS' within our runs; compare 'LPIPS(dk-conv)' against published values.")

## 5. Paired Wilcoxon (PhysIR vs actual DarkIR-m)

In [ ]:
import subprocess, itertools
if not HAVE_WILCOXON:
    print('paired_wilcoxon.py not in physir-code — upload it to run this cell.')
else:
    have = {f[len('per_image_'):-4]: os.path.join(PERIMG, f)
            for f in os.listdir(PERIMG) if f.startswith('per_image_')}
    # headline pairs (guide G6); only runs the ones whose CSVs are complete
    PAIRS = [('physir_full', 'darkir_m_released'),
             ('abl_percep384', 'darkir_m_released'),
             ('physir_full', 'abl_percep384')]
    # fall back to uploaded GPU CSV names if the CPU ones aren't done
    ALIAS = {'physir_full': ['physir_full', 'stage4_physir_final'],
             'abl_percep384': ['abl_percep384', 'stage5_percep'],
             'darkir_m_released': ['darkir_m_released']}
    def resolve(lab):
        for a in ALIAS.get(lab, [lab]):
            if a in have and sum(1 for _ in open(have[a])) - 1 >= N_TEST:
                return a, have[a]
        return None, None
    for a, b in PAIRS:
        (na, pa), (nb, pb) = resolve(a), resolve(b)
        if not pa or not pb:
            print(f'skip {a} vs {b} — CSV incomplete'); continue
        print(f'\n================ {na}  vs  {nb} ================')
        subprocess.run(['python', os.path.join(CODE_ROOT, 'paired_wilcoxon.py'),
                        pa, pb, '--name-a', na, '--name-b', nb])